# Project 2: Digital Advertising Budget Optimizer

**Big Data Analytics | Complete, reproducible Google Colab project**  
**Management question:** *Where should the company spend its next Rs. 1,000,000?*

This notebook implements the uploaded project brief end-to-end:

| Requirement in brief | Implementation |
|---|---|
| Facebook, Instagram, Google, TikTok, YouTube | Real source platform lookup; all five evaluated |
| `date, campaign, platform, impressions, clicks, spend, conversions, revenue, age, gender, city, device` | Standardized 12-column schema and documented provenance |
| CTR, CPC, CPA, conversion rate, ROAS, revenue | Recomputed from **numerators and denominators** |
| Campaign performance | Campaign, objective, platform, audience, device, and trend summaries |
| 10k → 100k → 1 million → 10 million | Deterministic, chunked synthetic CSV generator |
| Pandas vs MapReduce vs Spark | Three equivalent per-platform aggregation implementations, timed and validated |
| Marketing Budget Recommendation Engine | Constrained, uncertainty-aware five-platform optimization of Rs. 1 million |

**Important distinction:** The source is a *synthetic educational dataset*, not live marketing outcomes. The project is an analytical demonstration, not a guarantee of advertising returns.

## 0 · Source dataset and methodological choices

**Selected open dataset:** [Digital Marketing Campaign Performance Analysis (Giftedman1, GitHub)](https://github.com/Giftedman1/digital-marketing-campaign-analysis) (MIT-licensed repository). The project has **10,000 campaign records across seven platforms**; this notebook filters it to the five platforms explicitly named in the assignment. Downloaded files:

- [`Dataset/campaigns.csv`](https://github.com/Giftedman1/digital-marketing-campaign-analysis/blob/main/Dataset/campaigns.csv): campaign records with dates, cost, revenue, impressions, clicks, conversions, and device targeting.
- [`Dataset/platforms.csv`](https://github.com/Giftedman1/digital-marketing-campaign-analysis/blob/main/Dataset/platforms.csv): IDs to platform names.
- [`Dataset/customers.csv`](https://github.com/Giftedman1/digital-marketing-campaign-analysis/blob/main/Dataset/customers.csv): an **unlinked** customer table for realistic demographic *sampling* only, not evidence about a particular campaign.

### What is observed vs. simulated?

| Required field | Origin / limitation |
|---|---|
| date | Campaign **start** date (`StartDate`), not a daily ad event date |
| campaign | `CampaignName`; unique identifier additionally preserved |
| platform | `PlatformID` joined to `PlatformName` |
| impressions / clicks / conversions | Source campaign totals |
| spend / revenue | Source `Cost` and `Revenue`; original monetary units are **USD** |
| age / gender | Simulated, drawn from an independent customer population; **not actual campaign audiences** |
| city | Simulated from source's broad geographic region; **no observed city-level data** |
| device | Source `DeviceTarget`; this is **targeting**, not measured device of conversions |

We exclude non-brief platforms (LinkedIn, X) but do not relabel them. We never claim simulated demographics are observed, and we **do not use city, age, or gender to drive budget choices**. For the rupee budget illustration, historical *ROAS* is dimensionless, so the optimizer applies those ratios to a PKR budget under an explicitly stated *transferability assumption*. No USD→PKR conversion is claimed.

**Important**: 10 million rows is a **simulation** and is excluded from Run all by default on free Colab RAM/CPU. Set `RUN_10_MILLION=True` below to generate and test that scale. Benchmark results are never pre-invented.

In [ ]:
# 1. Imports and controls. Change ONLY these settings for an initial run.
from pathlib import Path
import os, sys, csv, io, gc, time, shutil, warnings, subprocess, importlib.util
from itertools import islice
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

SEED = 2026
BUDGET_PKR = 1_000_000
SOURCE_MODE = "github"                  # "github" or "synthetic" for a fully offline demonstration
ALLOW_SYNTHETIC_FALLBACK = True          # if GitHub download is unavailable, print a visible warning
RUN_10_MILLION = False                  # set True to include 10 million rows in generation + benchmarks
RUN_SPARK = True                       # Spark install and benchmark; False for an offline/no-Java check
KEEP_LARGE_DATASETS = False            # remove 100k+/1M+/10M CSV files after benchmarks
CHUNK_SIZE = 100_000                   # bounded-memory generation
BOOTSTRAP_REPS = 200
MIN_SHARE, MAX_SHARE = 0.05, 0.45      # min 5%, max 45% per platform
DIMINISHING_RETURNS = 0.30            # assumed, NOT estimated from this cross-sectional dataset
BUDGET_STEP_PKR = 5_000                # exact budget increments; total is divisible by this

WORK_DIR = Path('/content/advertising_budget_optimizer') if Path('/content').exists() else Path.cwd() / 'advertising_budget_optimizer'
DATA_DIR = WORK_DIR / 'data'
RESULTS_DIR = WORK_DIR / 'results'
for folder in (WORK_DIR, DATA_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PLATFORMS = ['Facebook', 'Instagram', 'Google Ads', 'TikTok', 'YouTube']
CORE_COLUMNS = ['date','campaign','platform','impressions','clicks','spend',
                'conversions','revenue','age','gender','city','device']
VALUE_COLUMNS = ['impressions','clicks','conversions','spend','revenue']
print('Working directory:', WORK_DIR)
print('5 platforms:', ', '.join(PLATFORMS))

## 1 · Download and inspect the source

Source files are cached in `data/source_*.csv`, so rerunning cells does not redownload them. If downloading fails, the notebook **clearly labels** its fallback as *independently generated synthetic data*.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/Giftedman1/digital-marketing-campaign-analysis/main/Dataset/'
FILES = {'campaigns': 'campaigns.csv', 'platforms': 'platforms.csv', 'customers': 'customers.csv'}

def fetch_csv(name, basename):
    path = DATA_DIR / f'source_{basename}'
    if not path.exists():
        import urllib.request
        req = urllib.request.Request(BASE_URL + basename, headers={'User-Agent':'GoogleColab-DataScience-Project'})
        with urllib.request.urlopen(req, timeout=35) as response, path.open('wb') as target:
            shutil.copyfileobj(response, target)
    return pd.read_csv(path, low_memory=False)

def make_fallback_source(n=10_000, seed=SEED):
    """Original, unrelated illustrative simulation; NEVER presented as GitHub observations."""
    rng = np.random.default_rng(seed)
    pid = rng.choice([1,2,3,4,5], n, p=[.20,.21,.20,.20,.19])
    profiles = {1:(.030, .058, 2.5, 130.), 2:(.025,.052,1.5,84.),
                3:(.020,.049,1.8,104.), 4:(.018,.047,3.0,150.), 5:(.015,.045,1.2,72.)}
    impressions = np.maximum(10, rng.lognormal(np.log(650_000), .65, n).astype('int64'))
    ctr = np.array([profiles[int(k)][0] for k in pid]) * rng.lognormal(0,.12,n)
    clicks = rng.binomial(impressions, np.clip(ctr,0,.9)).astype('int64')
    cvr = np.array([profiles[int(k)][1] for k in pid]) * rng.lognormal(0,.16,n)
    conversions = rng.binomial(clicks, np.clip(cvr,0,.9)).astype('int64')
    cpc = np.array([profiles[int(k)][2] for k in pid]) * rng.lognormal(0,.11,n)
    aov = np.array([profiles[int(k)][3] for k in pid]) * rng.lognormal(0,.19,n)
    date = pd.Timestamp('2024-08-16') + pd.to_timedelta(rng.integers(0,731,n),unit='D')
    objective = rng.choice(['Conversion','Lead Generation','Brand Awareness','Retention'],n)
    campaigns = pd.DataFrame({
        'CampaignID':[f'DEMO{i:06d}' for i in range(n)],
        'CampaignName':[f'{objective[i]} | Demo {i%70:02d}' for i in range(n)],
        'PlatformID':pid,
        'CampaignObjective':objective,
        'StartDate':date.strftime('%Y-%m-%d'),
        'Cost':np.round(clicks*cpc,2), 'Revenue':np.round(conversions*aov,2),
        'Impressions':impressions,'Clicks':clicks,'Conversions':conversions,
        'DeviceTarget':rng.choice(['Mobile','Desktop','Tablet','All'],n,p=[.47,.32,.12,.09]),
        'Region':rng.choice(['Asia-Pacific','North America','Europe','Global','Africa'],n),
    })
    pl = pd.DataFrame({'PlatformID':[1,2,3,4,5],
                       'PlatformName':['Google Ads','TikTok','Instagram','YouTube','Facebook']})
    customers = pd.DataFrame({'Age':rng.integers(18,66,5_000),
                              'Gender':rng.choice(['Female','Male','Other'],5_000,p=[.50,.48,.02])})
    return campaigns, pl, customers

if SOURCE_MODE == 'github':
    try:
        raw_campaigns, raw_platforms, raw_customers = (
            fetch_csv('campaigns',FILES['campaigns']),
            fetch_csv('platforms',FILES['platforms']),
            fetch_csv('customers',FILES['customers']))
        DATA_PROVENANCE = 'GITHUB MIT-licensed SYNTHETIC campaign dataset'
    except Exception as exc:
        if not ALLOW_SYNTHETIC_FALLBACK:
            raise RuntimeError('Source dataset unavailable. Provide source_campaigns.csv etc.') from exc
        warnings.warn('SOURCE DOWNLOAD FAILED: using INDEPENDENT SIMULATED FALLBACK, not the GitHub dataset. '
                      f'Reason: {type(exc).__name__}: {exc}', RuntimeWarning)
        raw_campaigns, raw_platforms, raw_customers = make_fallback_source()
        DATA_PROVENANCE = 'OFFLINE INDEPENDENTLY SIMULATED FALLBACK (NOT GITHUB DATA)'
elif SOURCE_MODE == 'synthetic':
    raw_campaigns, raw_platforms, raw_customers = make_fallback_source()
    DATA_PROVENANCE = 'OFFLINE INDEPENDENTLY SIMULATED FALLBACK (NOT GITHUB DATA)'
else:
    raise ValueError('SOURCE_MODE must be github or synthetic')

print('DATA PROVENANCE:', DATA_PROVENANCE)
print('Raw campaigns:',len(raw_campaigns),'rows | columns:',len(raw_campaigns.columns))
display(raw_campaigns.head(3))

## 2 · Prepare the required twelve-column schema and audit data quality

Numeric outliers are not silently deleted. Recompute KPIs from the five primitive counts and totals; in particular **CPA is undefined when there are zero conversions** (not zero). Filtering removes invalid rows and out-of-scope platforms. `CampaignID` remains available for campaign-level reporting.

In [ ]:
REGION_CITY_MAP = {
    'Asia-Pacific':['Karachi','Lahore','Mumbai','Singapore'],
    'North America':['New York','Chicago','Toronto','Los Angeles'],
    'Europe':['London','Paris','Berlin','Amsterdam'],
    'Africa':['Lagos','Nairobi','Cairo','Cape Town'],
    'Middle East':['Dubai','Riyadh','Doha','Abu Dhabi'],
    'Latin America':['Bogota','Lima','Sao Paulo','Mexico City'],
    'South America':['Bogota','Lima','Sao Paulo','Santiago'],
    'Global':['Simulated Global City A','Simulated Global City B']
}

def normalize_and_audit(campaigns, platforms, customers, seed=SEED):
    needed=['CampaignID','CampaignName','PlatformID','StartDate','Cost','Revenue',
            'Impressions','Clicks','Conversions','DeviceTarget','Region']
    missing=sorted(set(needed)-set(campaigns.columns))
    if missing: raise ValueError(f'Missing source campaign columns: {missing}')
    lookup=platforms[['PlatformID','PlatformName']].drop_duplicates('PlatformID')
    df=campaigns.merge(lookup,how='left',on='PlatformID',validate='many_to_one')
    df=df.rename(columns={
        'CampaignID':'campaign_id','CampaignName':'campaign','StartDate':'date',
        'PlatformName':'platform','Cost':'spend','Revenue':'revenue',
        'Impressions':'impressions','Clicks':'clicks','Conversions':'conversions',
        'DeviceTarget':'device','Region':'region','CampaignObjective':'objective'
    })
    df['date']=pd.to_datetime(df['date'],errors='coerce')
    for c in VALUE_COLUMNS: df[c]=pd.to_numeric(df[c],errors='coerce')
    before=len(df)
    duplicates=df['campaign_id'].duplicated(keep='first').sum()
    is_five=df['platform'].isin(PLATFORMS)
    is_valid=(df['date'].notna() & df[VALUE_COLUMNS].notna().all(axis=1)
              & np.isfinite(df[VALUE_COLUMNS].to_numpy(dtype=float)).all(axis=1)
              & (df[VALUE_COLUMNS]>=0).all(axis=1)
              & (df['clicks']<=df['impressions']) & (df['conversions']<=df['clicks'])
              & df['campaign_id'].notna())
    report = pd.DataFrame({'audit_item':['Source campaign records','Out-of-scope platforms',
                                         'Invalid metric/date/id records','Duplicate campaign IDs',
                                         'Accepted five-platform records'],
                           'count':[before,int((~is_five).sum()),int((~is_valid).sum()),
                                    int(duplicates),None]})
    df=df.loc[is_five & is_valid].drop_duplicates('campaign_id').copy()
    report.loc[4,'count']=len(df)
    if not set(PLATFORMS).issubset(set(df['platform'])):
        raise ValueError('At least one required platform missing; cannot make a five-way recommendation')
    for c in ('impressions','clicks','conversions'):
        df[c]=df[c].astype('int64')
    df['date']=df['date'].dt.strftime('%Y-%m-%d')
    if 'objective' not in df.columns: df['objective']='Unspecified'
    df['device']=df['device'].fillna('Unknown').astype(str)
    df['region']=df['region'].fillna('Global').astype(str)
    df['campaign']=df['campaign'].fillna('Unnamed campaign').astype(str)
    customer_ages=pd.to_numeric(customers.get('Age',pd.Series([30])),errors='coerce').dropna()
    customer_ages=customer_ages[(customer_ages>=18)&(customer_ages<=100)]
    if customer_ages.empty: customer_ages=pd.Series([30])
    genders=customers.get('Gender',pd.Series(['Unknown'])).dropna().astype(str)
    if genders.empty: genders=pd.Series(['Unknown'])
    rng=np.random.default_rng(seed)
    # Demographics are sampled *independently* and do NOT represent campaign measurements.
    df['age']=rng.choice(customer_ages.to_numpy(dtype=int),size=len(df),replace=True)
    df['gender']=rng.choice(genders.to_numpy(),size=len(df),replace=True)
    df['city']=[str(rng.choice(REGION_CITY_MAP.get(r, ['Simulated Other City'])))
                for r in df['region'].to_numpy()]
    df['demographics_simulated']=True
    df['city_simulated']=True
    df=df[CORE_COLUMNS+['campaign_id','objective','region','demographics_simulated','city_simulated']]
    df=df.reset_index(drop=True)
    return df,report

base, audit = normalize_and_audit(raw_campaigns, raw_platforms, raw_customers)
print('DATA PROVENANCE:', DATA_PROVENANCE)
display(audit)
print('Required column check:', all(c in base for c in CORE_COLUMNS))
display(base.head(5))
print('Platforms:',base['platform'].value_counts().to_dict())
base.to_csv(DATA_DIR/'cleaned_five_platforms.csv', index=False)

## 3 · KPIs: weighted arithmetic, not averages of precomputed percentages

All percentage values below are *aggregate-weighted*:

\[CTR=100\frac{\sum clicks}{\sum impressions},\quad CVR=100\frac{\sum conversions}{\sum clicks}\]
\[CPC=\frac{\sum spend}{\sum clicks},\quad CPA=\frac{\sum spend}{\sum conversions},\quad ROAS=\frac{\sum revenue}{\sum spend}\]

**Source cost/revenue/cost per click are in USD**, while the later management budget is in PKR. We only transfer *dimensionless ROAS* to the PKR what-if; we do not mix USD CPA or USD CPC with rupees.

In [ ]:
def safe_divide(numerator, denominator):
    """Undefined ratios are NaN; no false zero CPC or CPA when denominator is 0."""
    n=np.asarray(numerator,dtype=float)
    d=np.asarray(denominator,dtype=float)
    out=np.full(np.broadcast_shapes(n.shape,d.shape), np.nan, dtype=float)
    np.divide(n,d,out=out,where=(d!=0))
    return out.item() if out.ndim==0 else out

def summarize_kpis(df, group):
    stats=df.groupby(group,dropna=False,observed=True)[VALUE_COLUMNS].sum().reset_index()
    stats['CTR_pct']=100*safe_divide(stats['clicks'],stats['impressions'])
    stats['CPC_usd']=safe_divide(stats['spend'],stats['clicks'])
    stats['CPA_usd']=safe_divide(stats['spend'],stats['conversions'])
    stats['CVR_pct']=100*safe_divide(stats['conversions'],stats['clicks'])
    stats['ROAS']=safe_divide(stats['revenue'],stats['spend'])
    stats['profit_usd']=stats['revenue']-stats['spend']
    return stats

platform_kpis=summarize_kpis(base,'platform').set_index('platform').reindex(PLATFORMS).reset_index()
objective_kpis=summarize_kpis(base,'objective').sort_values('revenue',ascending=False)
campaign_kpis=summarize_kpis(base,'campaign_id').sort_values('revenue',ascending=False)
overall=base[VALUE_COLUMNS].sum()
overall_roas=overall['revenue']/overall['spend']
print('Historical source overview:', DATA_PROVENANCE)
print(f"Campaigns: {len(base):,} | Spend: ${overall['spend']:,.2f} | Revenue: ${overall['revenue']:,.2f} | ROAS: {overall_roas:.2f}x")
display(platform_kpis.round(3))
print('Top campaign objectives by revenue:')
display(objective_kpis.round(2))
print('Top 8 campaigns by revenue:')
display(campaign_kpis.head(8).round(2))
platform_kpis.to_csv(RESULTS_DIR/'platform_kpis.csv',index=False)
objective_kpis.to_csv(RESULTS_DIR/'objective_kpis.csv',index=False)

In [ ]:
# Data quality checks, including zero denominators and expected weighted CTR.
toy=pd.DataFrame({'platform':['Facebook','Facebook'],'impressions':[100,0],
                  'clicks':[10,0],'conversions':[0,0], 'spend':[20.,0.], 'revenue':[40.,0.]})
toy_kpi=summarize_kpis(toy,'platform').iloc[0]
assert np.isclose(toy_kpi['CTR_pct'],10.0)
assert np.isnan(toy_kpi['CPA_usd'])
assert np.isclose(toy_kpi['ROAS'],2.0)
assert (base['conversions'] <= base['clicks']).all()
assert (base['clicks'] <= base['impressions']).all()
assert base['date'].notna().all()
print('KPI unit checks passed; zero-conversion CPA is undefined.')

## 4 · Campaign / audience / time diagnostics and figures

Figures show **associations in the synthetic source**, not causal effects. In particular, simulated age/gender/city cannot justify targeting recommendations; the device field denotes device targeting.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
rank=platform_kpis.sort_values('ROAS',ascending=True)
axes[0].barh(rank['platform'],rank['ROAS'])
axes[0].set_xlabel('Revenue / ad spend (x)')
axes[0].set_title('Historical ROAS by platform')
for i,val in enumerate(rank['ROAS']): axes[0].text(val+0.015,i,f'{val:.2f}x',va='center')
axes[0].set_xlim(0,max(rank['ROAS'])*1.18)
axes[1].bar(platform_kpis['platform'],platform_kpis['spend']/1e6,label='Spend')
axes[1].plot(platform_kpis['platform'],platform_kpis['revenue']/1e6,'o-',label='Revenue')
axes[1].tick_params(axis='x',rotation=40)
axes[1].set_ylabel('Source USD, millions')
axes[1].set_title('Total cost and attributed revenue')
axes[1].legend()
plt.tight_layout(); plt.show()

month_df=base.assign(month=pd.to_datetime(base['date']).dt.to_period('M').astype(str))
monthly=summarize_kpis(month_df,'month')
fig,ax=plt.subplots(figsize=(12,3.5))
ax.plot(monthly['month'],monthly['ROAS'],marker='o',markersize=3)
ax.set_title('Monthly ROAS by campaign start month (not daily spend)')
ax.set_ylabel('ROAS (x)');ax.tick_params(axis='x',rotation=90)
plt.tight_layout(); plt.show()

for field in ['device','age','gender']:
    grouped=summarize_kpis(base,field)
    if field=='age':
        grouped['age_band']=pd.cut(grouped[field],bins=[17,24,34,44,54,64,100],
                                   labels=['18–24','25–34','35–44','45–54','55–64','65+'])
        grouped=summarize_kpis(base.assign(age_band=pd.cut(base['age'],[17,24,34,44,54,64,100],
            labels=['18–24','25–34','35–44','45–54','55–64','65+'])),'age_band')
    print(f'{field}:', 'SIMULATED, NOT EMPIRICAL' if field in ['age','gender'] else 'TARGETING, NOT OBSERVED')
    display(grouped[[('age_band' if field=='age' else field),'spend','conversions','ROAS']].round(2).head(12))

print('CITY WARNING: every city is modeled from broad regional tags; NO observed city performance conclusions.')
display(base[['region','city','city_simulated']].drop_duplicates().head(10))

## 5 · Big Data simulation to 10k, 100k, 1m and 10m rows

Rather than duplicating every row, we **bootstrap campaign records**, perturb impressions/CTR/CVR/CPC/order value, and resample clicks and conversions as conditional binomial events. Relationships are preserved: `0 ≤ conversions ≤ clicks ≤ impressions`, `spend ≥ 0`, `revenue ≥ 0`. Data are created and written in **100k-row chunks**, so the 10m mode does not require constructing a 10m-row DataFrame in memory. These **are synthetic rows**, not 10 million real campaigns.

In [ ]:
def build_simulation_cache(base_df):
    b=base_df.copy().reset_index(drop=True)
    b['ctr_base']=safe_divide(b['clicks'],b['impressions'])
    b['cvr_base']=safe_divide(b['conversions'],b['clicks'])
    b['cpc_base']=safe_divide(b['spend'],b['clicks'])
    b['aov_base']=safe_divide(b['revenue'],b['conversions'])
    b['ctr_base']=b['ctr_base'].fillna(0.015).clip(0.000001,.25)
    b['cvr_base']=b['cvr_base'].fillna(0.03).clip(0,.5)
    for c,fallback in [('cpc_base',2.0),('aov_base',100.0)]:
        median=b.loc[(b[c]>0)&np.isfinite(b[c]),c].median()
        if not np.isfinite(median): median=fallback
        b[c]=b[c].where(np.isfinite(b[c])&(b[c]>0),median)
    return b

SIM_BASE=build_simulation_cache(base)

def simulate_chunk(n, rng, start=0, model=SIM_BASE):
    idx=rng.integers(0,len(model),size=n)
    sampled=model.iloc[idx]
    count=sampled['impressions'].to_numpy(dtype='int64')
    scale=rng.lognormal(mean=-0.07,sigma=.35,size=n)
    impressions=np.maximum(1,np.rint(count*scale).astype('int64'))
    ctr=np.clip(sampled['ctr_base'].to_numpy()*rng.lognormal(0,.14,n),0,.60)
    clicks=rng.binomial(impressions,ctr).astype('int64')
    cvr=np.clip(sampled['cvr_base'].to_numpy()*rng.lognormal(0,.14,n),0,.75)
    conversions=rng.binomial(clicks,cvr).astype('int64')
    cpc=sampled['cpc_base'].to_numpy()*rng.lognormal(0,.10,n)
    aov=sampled['aov_base'].to_numpy()*rng.lognormal(0,.15,n)
    out=pd.DataFrame({
        'date':sampled['date'].to_numpy(),
        'campaign':sampled['campaign'].to_numpy(),
        'platform':sampled['platform'].to_numpy(),
        'impressions':impressions,'clicks':clicks,
        'spend':np.round(clicks*cpc,2), 'conversions':conversions,
        'revenue':np.round(conversions*aov,2),
        'age':sampled['age'].to_numpy(),
        'gender':sampled['gender'].to_numpy(),
        'city':sampled['city'].to_numpy(),
        'device':sampled['device'].to_numpy()
    })
    assert len(out)==n
    assert (out['clicks']<=out['impressions']).all()
    assert (out['conversions']<=out['clicks']).all()
    assert out[['spend','revenue']].ge(0).all().all()
    return out[CORE_COLUMNS]

def generate_csv(n,path,seed=SEED,chunk_size=CHUNK_SIZE):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(seed + n)
    begun=time.perf_counter()
    for start in range(0,n,chunk_size):
        batch=simulate_chunk(min(chunk_size,n-start),rng,start=start)
        batch.to_csv(path,mode='w' if start==0 else 'a',index=False,header=(start==0))
    took=time.perf_counter()-begun
    print(f'{n:,} synthetic records -> {path.name}: {path.stat().st_size/1e6:.1f} MB, generated in {took:.1f}s')
    return path

sim10k_path=generate_csv(10_000, DATA_DIR/'simulated_10k.csv')
sim10k=pd.read_csv(sim10k_path)
assert len(sim10k)==10_000
assert set(CORE_COLUMNS)==set(sim10k.columns)
print('Synthetic 10k data sample:')
display(sim10k.head())
print('Source ROAS vs 10k simulated ROAS:')
display(pd.DataFrame({'historical':platform_kpis.set_index('platform')['ROAS'],
                     'simulated':summarize_kpis(sim10k,'platform').set_index('platform')['ROAS']}).round(3))

## 6 · Budget Recommendation Engine: Rs. 1 million

**Goal:** allocate Rs. 1,000,000 across **all five** platforms based on historical, USD-denominated ROAS ratios (unit-free) while accounting for sampling variability and diminishing returns. We use a **bootstrap lower bound** to make a modestly conservative score rather than trusting a single point estimate.

The explicit hypothetical response curve is:

\[\widehat{Revenue}_i(B_i) = q_i B_i \left(1 - \lambda \frac{B_i}{1{,}000{,}000}\right)\]

where `qᵢ = 0.7 × source ROAS + 0.3 × 10th-percentile bootstrapped ROAS`, `λ = 0.30` (user-chosen assumption). Constraints: at least 5%, at most 45% on any platform, increments of Rs. 5,000. We use a **greedy marginal-gain allocator** that is optimal for this discretized concave objective with separable per-channel curves. The baseline is the actual historical *spend mix* applied to Rs. 1 million. Predicted improvement is an illustrative model comparison, **not validated incremental lift**.

In [ ]:
def bootstrap_roas(df, platforms=PLATFORMS, reps=BOOTSTRAP_REPS, seed=SEED):
    rng=np.random.default_rng(seed+11)
    ci=[]
    for p in platforms:
        sub=df.loc[df['platform']==p, ['spend','revenue']].to_numpy(dtype='float64')
        idx=rng.integers(0,len(sub),size=(reps,len(sub)))
        spends=sub[idx,0].sum(axis=1); revenue=sub[idx,1].sum(axis=1)
        samples=np.divide(revenue,spends,out=np.zeros_like(spends),where=spends>0)
        ci.append({'platform':p,'roas_ci10':np.quantile(samples,.10),
                   'roas_ci50':np.quantile(samples,.50),'roas_ci90':np.quantile(samples,.90)})
    return pd.DataFrame(ci)

uncertainty=bootstrap_roas(base)
budget_scores=platform_kpis[['platform','spend','ROAS','CPA_usd','CVR_pct']].merge(uncertainty,on='platform')
budget_scores['conservative_roas']=.70*budget_scores['ROAS']+.30*budget_scores['roas_ci10']
budget_scores['historical_share']=budget_scores['spend']/budget_scores['spend'].sum()

# Validate constraints so the optimizer cannot silently produce an infeasible recommendation.
assert BUDGET_PKR%BUDGET_STEP_PKR==0
assert MIN_SHARE*len(PLATFORMS) <= 1 <= MAX_SHARE*len(PLATFORMS)
assert (MIN_SHARE*BUDGET_PKR)%BUDGET_STEP_PKR == 0
assert (MAX_SHARE*BUDGET_PKR)%BUDGET_STEP_PKR == 0

def modeled_revenue(shares, scores, decay=DIMINISHING_RETURNS, budget=BUDGET_PKR):
    shares=np.asarray(shares,dtype=float)
    q=np.asarray(scores,dtype=float)
    return float(np.sum(q*budget*shares*(1-decay*shares)))

def optimize_shares(scores,decay=DIMINISHING_RETURNS,
                    min_share=MIN_SHARE,max_share=MAX_SHARE):
    scores=np.asarray(scores,dtype='float64')
    total_units=int(BUDGET_PKR//BUDGET_STEP_PKR)
    min_units=int(round(min_share*total_units)); max_units=int(round(max_share*total_units))
    units=np.full(len(scores),min_units,dtype='int64')
    assert units.sum()<=total_units<=max_units*len(units)
    # Incremental gain for moving share s -> s+delta; curves concave for decay>=0.
    step=1.0/total_units
    while units.sum()<total_units:
        shares=units/total_units
        gains=scores*BUDGET_PKR*(step - decay*(2*shares*step+step**2))
        gains=np.where(units<max_units,gains,-np.inf)
        best=int(np.argmax(gains))
        if not np.isfinite(gains[best]):raise RuntimeError('No feasible platform can take another budget increment')
        units[best]+=1
    shares=units/total_units
    assert np.isclose(shares.sum(),1.)
    assert shares.min()>=min_share-1e-12 and shares.max()<=max_share+1e-12
    return shares

scores=budget_scores['conservative_roas'].to_numpy()
new_shares=optimize_shares(scores)
old_shares=budget_scores['historical_share'].to_numpy()
projected_baseline=modeled_revenue(old_shares,scores)
projected_optimized=modeled_revenue(new_shares,scores)
budget_scores['recommended_share']=new_shares
budget_scores['recommendation_pct']=np.round(100*new_shares,2)
budget_scores['allocated_PKR']=(new_shares*BUDGET_PKR).round().astype('int64')
budget_scores['modeled_revenue_PKR']=scores*budget_scores['allocated_PKR']*(1-DIMINISHING_RETURNS*new_shares)
budget_scores['delta_share_pp']=100*(new_shares-old_shares)
assert budget_scores['allocated_PKR'].sum()==BUDGET_PKR

print('MANAGEMENT DECISION: where to spend Rs. 1,000,000')
display(budget_scores[['platform','ROAS','roas_ci10','roas_ci90',
                       'historical_share','recommendation_pct','allocated_PKR','delta_share_pp',
                       'modeled_revenue_PKR']].round(3).sort_values('allocated_PKR',ascending=False))
print(f'Historical-mix modeled revenue: Rs. {projected_baseline:,.0f}')
print(f'Optimized-mix modeled revenue: Rs. {projected_optimized:,.0f}')
print(f'Illustrative modeled gain: {100*(projected_optimized/projected_baseline-1):.2f}% (NOT causal or verified)')
budget_scores.to_csv(RESULTS_DIR/'budget_recommendation_1m_PKR.csv',index=False)

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.8))
x=np.arange(len(budget_scores)); w=.38
ax.bar(x-w/2,100*budget_scores['historical_share'],w,label='Historical spend share')
ax.bar(x+w/2,budget_scores['recommendation_pct'],w,label='Recommended share')
ax.set_xticks(x,budget_scores['platform'],rotation=20)
ax.set_ylabel('% of Rs. 1,000,000')
ax.set_title('Five-platform budget recommendation; 5% to 45% per platform')
ax.legend(); plt.tight_layout();plt.show()

sensitivity=[]
for decay in [0.,.15,.30,.50]:
    s=optimize_shares(scores,decay=decay)
    for p,share in zip(budget_scores['platform'],s):
        sensitivity.append({'assumed_decay':decay,'platform':p,'share_pct':share*100})
sensitivity=pd.DataFrame(sensitivity)
print('Sensitivity to diminishing-return assumption:')
display(sensitivity.pivot(index='platform',columns='assumed_decay',values='share_pct').round(1))
sensitivity.to_csv(RESULTS_DIR/'budget_sensitivity.csv',index=False)

## 7 · Pandas vs. MapReduce vs. Spark: same CSV files, same group totals

These methods calculate the identical five per-platform **sums** from each generated CSV, **including disk read time** (but excluding data generation time):

1. **Pandas** in-memory `read_csv` plus `groupby`.
2. **Python MapReduce**: CSV stream divided into fixed-size chunks; **map** each chunk to partial key/value sums and **reduce** the partials into final platform sums. This is a *single-computer educational MapReduce simulation*, **not a Hadoop cluster**.
3. **Apache Spark**: genuine PySpark DataFrame `groupBy` with local executors inside Colab, **not a multi-node cluster**.

The Spark session is started once and its startup time is not included in benchmark time. Results depend on hardware, file size, caching, CSV parsing, and single-run noise; no method is guaranteed to win. `RUN_10_MILLION=True` enables the full 10m-row test and may require a high-RAM runtime.

In [ ]:
# Benchmark implementations: all return identical columns and results.
def canonical_agg(frame):
    out=frame.copy()
    out.index=out.index.astype(str)
    return out.reindex(PLATFORMS).fillna(0)[VALUE_COLUMNS].astype(float)

def bench_pandas(path):
    dtype={'platform':'category','impressions':'int64','clicks':'int64',
           'conversions':'int64','spend':'float64','revenue':'float64'}
    frame=pd.read_csv(path,usecols=['platform']+VALUE_COLUMNS,dtype=dtype)
    aggregated=frame.groupby('platform',observed=True)[VALUE_COLUMNS].sum()
    del frame
    return canonical_agg(aggregated)

def map_records(chunk):
    """Map one bounded partition to key -> [five sums]."""
    partial={}
    for r in chunk:
        key=r['platform']
        accumulator=partial.setdefault(key,[0,0,0,0.,0.])
        accumulator[0]+=int(r['impressions'])
        accumulator[1]+=int(r['clicks'])
        accumulator[2]+=int(r['conversions'])
        accumulator[3]+=float(r['spend'])
        accumulator[4]+=float(r['revenue'])
    return partial

def reduce_partials(accumulator,partial):
    """Reduce partial platform sums into a global result."""
    for key,vals in partial.items():
        target=accumulator.setdefault(key,[0,0,0,0.,0.])
        for k,v in enumerate(vals): target[k]+=v
    return accumulator

def bench_mapreduce(path,block_size=25_000):
    reduced={}
    with open(path,'r',newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        while True:
            chunk=list(islice(reader,block_size))
            if not chunk:break
            reduce_partials(reduced,map_records(chunk))
    aggregated=pd.DataFrame.from_dict(reduced,orient='index',columns=VALUE_COLUMNS)
    return canonical_agg(aggregated)

spark=None
if RUN_SPARK:
    if not importlib.util.find_spec('pyspark'):
        try:
            print('Installing PySpark (the first run may download a substantial package)...')
            subprocess.check_call([sys.executable,'-m','pip','install','-q','pyspark>=3.5,<4.2'])
        except Exception as exc:
            warnings.warn(f'PySpark installation failed; Spark comparison skipped: {exc}')
            RUN_SPARK=False
    if RUN_SPARK:
        try:
            from pyspark.sql import SparkSession, functions as F, types as T
            spark=(SparkSession.builder.master('local[2]')
                   .appName('DigitalAdvertisingBudgetOptimizer')
                   .config('spark.sql.shuffle.partitions','4')
                   .config('spark.ui.enabled','false')
                   .getOrCreate())
            spark.sparkContext.setLogLevel('ERROR')
            print('Spark version:',spark.version)
        except Exception as exc:
            warnings.warn(f'Spark initialization failed; Spark comparison skipped: {exc}')
            RUN_SPARK=False

if RUN_SPARK:
    BENCH_SCHEMA=T.StructType([
        T.StructField('date',T.StringType()),
        T.StructField('campaign',T.StringType()),
        T.StructField('platform',T.StringType()),
        T.StructField('impressions',T.LongType()),
        T.StructField('clicks',T.LongType()),
        T.StructField('spend',T.DoubleType()),
        T.StructField('conversions',T.LongType()),
        T.StructField('revenue',T.DoubleType()),
        T.StructField('age',T.IntegerType()),
        T.StructField('gender',T.StringType()),
        T.StructField('city',T.StringType()),
        T.StructField('device',T.StringType())])

def bench_spark(path):
    if not RUN_SPARK:raise RuntimeError('Spark not available')
    source=spark.read.option('header','true').schema(BENCH_SCHEMA).csv(str(path))
    columns=[F.sum(F.col(c)).alias(c) for c in VALUE_COLUMNS]
    result=source.groupBy('platform').agg(*columns).toPandas().set_index('platform')
    return canonical_agg(result)

# A tiny correctness smoke test before scaling, including MapReduce parity.
smoke_pd=bench_pandas(sim10k_path)
smoke_mr=bench_mapreduce(sim10k_path)
np.testing.assert_allclose(smoke_pd.to_numpy(),smoke_mr.to_numpy(),rtol=1e-10,atol=.05)
if RUN_SPARK:
    smoke_sp=bench_spark(sim10k_path)
    np.testing.assert_allclose(smoke_pd.to_numpy(),smoke_sp.to_numpy(),rtol=1e-10,atol=.05)
print('All available aggregation implementations agree on the 10k smoke test.')

In [ ]:
SCALES=[10_000,100_000,1_000_000]
if RUN_10_MILLION:SCALES.append(10_000_000)
results=[]
for n in SCALES:
    path=sim10k_path if n==10_000 else generate_csv(n,DATA_DIR/f'simulated_{n}.csv')
    reference=None
    methods=[('Pandas',bench_pandas),('MapReduce (Python)',bench_mapreduce)]
    if RUN_SPARK:methods.append(('Spark (local)',bench_spark))
    for method,fn in methods:
        gc.collect()
        begun=time.perf_counter()
        agg=fn(path)
        elapsed=time.perf_counter()-begun
        if reference is None:reference=agg
        else: np.testing.assert_allclose(reference.to_numpy(),agg.to_numpy(),rtol=1e-10,atol=.05)
        results.append({'records':n,'method':method,'seconds':elapsed,
                        'csv_MB':path.stat().st_size/1e6, 'validated':True})
        print(f'{n:>10,}  {method:<20} {elapsed:>9.2f} seconds   VALIDATED')
    if n!=10_000 and not KEEP_LARGE_DATASETS:
        path.unlink(missing_ok=True)

benchmarks=pd.DataFrame(results)
benchmarks.to_csv(RESULTS_DIR/'benchmarks.csv',index=False)
print('Comparison table (seconds, measured on THIS runtime):')
display(benchmarks.pivot(index='records',columns='method',values='seconds').round(2))
fig,ax=plt.subplots(figsize=(10,5))
for method,sub in benchmarks.groupby('method'):
    ax.plot(sub['records'],sub['seconds'],marker='o',label=method)
ax.set_xscale('log');ax.set_xlabel('Number of simulated rows (log scale)')
ax.set_ylabel('Elapsed processing time (seconds)')
ax.set_title('Pandas vs Python MapReduce vs local Spark (same CSV + same totals)')
ax.legend();ax.grid(axis='y',alpha=.25)
plt.tight_layout();plt.show()
if not RUN_10_MILLION:
    print('10 MILLION ROWS NOT RUN IN THIS SESSION. Set RUN_10_MILLION=True and rerun this notebook.')

## 8 · Export results, validation and reproducibility

Results contain the **12-field simulated dataset**, the cleaned source slice, historical KPI tables, budget recommendations, sensitivity results, and measured benchmark results. Large-scale raw datasets are *not packaged by default*. To download the entire results ZIP in Google Colab, uncomment the two lines at the bottom of the cell.

In [ ]:
import json, zipfile
metadata={
 'project':'Project 2 - Digital Advertising Budget Optimizer',
 'dataset_provenance':DATA_PROVENANCE,
 'upstream_repo':'https://github.com/Giftedman1/digital-marketing-campaign-analysis',
 'upstream_license':'MIT as stated in source repository',
 'source_platforms':PLATFORMS,
 'source_rows_after_five_platform_filter':int(len(base)),
 'source_currency':'USD (source); PKR budget is an assumption-based what-if using unitless ROAS',
 'simulated_fields':['age','gender','city'],
 'device_field':'DeviceTarget; not observed device of converter',
 'scale_rows_supported':[10_000,100_000,1_000_000,10_000_000],
 'scale_rows_executed':SCALES,
 'spark_executed':bool(RUN_SPARK),
 'budget_pkr':BUDGET_PKR,
 'budget_steps_pkr':BUDGET_STEP_PKR,
 'diminishing_returns_assumption':DIMINISHING_RETURNS,
 'min_share':MIN_SHARE,'max_share':MAX_SHARE,
 'seed':SEED,
 'note':'Synthetic educational data and hypothetical budget; not measured incremental returns.'
}
with (RESULTS_DIR/'run_metadata.json').open('w') as f:
    json.dump(metadata,f,indent=2)

readme = f"""# Digital Advertising Budget Optimizer | Run outputs

DATA PROVENANCE: {DATA_PROVENANCE}

Source: {metadata['upstream_repo']}
The source is synthetic educational data, not actual historical advertising transactions.
Original monetary values are USD; the Rs. 1,000,000 allocation is a separate model scenario using dimensionless ROAS.
Age, gender, and city are simulated; device is device targeting.

Budget assumptions: 5-45% per platform; Rs. 5,000 steps; decay={DIMINISHING_RETURNS:.2f}.
The output lift is model-based, not a causal claim or measured prediction.

Benchmark methods: Pandas full-memory, Python local MapReduce simulation, and local PySpark if installed.
Executed scales: {SCALES}. Real multi-node Hadoop was not deployed.
"""
(RESULTS_DIR/'README.txt').write_text(readme)
package=WORK_DIR/'optimizer_run_results.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for path in sorted(RESULTS_DIR.iterdir()):
        if path.is_file():z.write(path,arcname='results/'+path.name)
    z.write(DATA_DIR/'simulated_10k.csv',arcname='data/simulated_10k.csv')
    z.write(DATA_DIR/'cleaned_five_platforms.csv',arcname='data/cleaned_five_platforms.csv')
print('Outputs packaged at:',package)
print('Included: simulated 10k, cleaned source slice, KPI tables, optimizer outputs, benchmarks, metadata.')
# from google.colab import files
# files.download(str(package))

## 9 · Final findings, defensible limitations and next steps

Once you've run the notebook, explain to management:

- **Recommendation:** spend the complete Rs. 1 million according to `budget_recommendation_1m_PKR.csv`; report each of the *five* platform percentages and rupee amounts. This is an *allocation scenario*.
- **Evidence:** compare actual source ROAS, CVR, CPA, revenue and campaign/objective breakdowns. All five platforms are represented by source campaign rows in the **synthetic** dataset.
- **Benchmarks:** report the observed elapsed times from your own Colab execution. The Python MapReduce example is **not Hadoop**, and local Spark is **not a cluster**.
- **Limitations:** no campaign-specific measured age/gender/city, no causal attribution, no validation of diminishing returns, no PKR transaction records, synthetic upstream data, and bootstrap intervals represent resampling variation **only**, not future business uncertainty.
- **For a real company:** replace the source with verified ad-platform exports in a consistent currency, collect conversion attribution and platform-specific marginal response tests, and only then deploy budget reallocation.

**External source and attribution:** [GitHub dataset and license](https://github.com/Giftedman1/digital-marketing-campaign-analysis). This notebook contains original educational processing and modeling code, not copied analysis notebooks.